### Demonstration of motion adjustment with CaImAn

In [2]:
# Auto-Reload of .py files
%reload_ext autoreload

import caiman_analysis as cma

##### First, determine a path to a file itself.

In [ ]:
fname = "./data/niels_video.avi" # path to video file
framerate = 5 # frames per second of the video

##### Load the video in CaImAn format.

In [ ]:
# Load video
m_orig = cma.load_video(fname=fname, fframe=framerate, play_movies=False, downsample_ratio=1, resize=False)

##### Determine the parameters for analysis.

In [ ]:
# Create a motion correction object with parameters for analysis
max_shifts = (12, 12)   # maximum allowed rigid shift in pixels (view the movie to get a sense of motion)
strides =  (96, 96)     # create a new patch every x pixels for pw-rigid correction
overlaps = (48, 48)     # overlap between patches (size of patch strides+overlaps)
max_deviation_rigid = 6 # maximum deviation allowed for patch with respect to rigid shifts
pw_rigid = False        # flag for performing rigid or piecewise rigid motion correction
shifts_opencv = True    # flag for correcting motion using bicubic interpolation (otherwise FFT interpolation is used)
border_nan = 'copy'     # replicate values along the boundary (if True, fill in with NaN)

##### Run the motion correction of that video.

In [13]:
# Run motion correction without pw_rigid correction
m_corr = cma.run_motioncorrect(fname=fname, max_shifts=max_shifts, strides=strides, overlaps=overlaps, max_deviation_rigid=max_deviation_rigid, 
                               shifts_opencv=shifts_opencv, downsample_ratio=1, nonneg_movie=True, border_nan=border_nan, save_movie=True, 
                               pw_rigid = False)

100%|██████████| 1/1 [00:00<00:00,  1.81it/s]


##### Export the motion corrected video as .avi file.

In [14]:
# Export of the file as .avi / .nwb / .tif
m_orig.save(file_name='./export/m_orig.avi', 
            q_min=0.0, 
            q_max=252.0)

m_corr.save(file_name='./export/m_corr.avi', 
            q_min=0.0, 
            q_max=252.0)

'./export/m_corr.avi'

### Motion adjustment of two videos

##### Determine the paths to the template and the file that needs adjustment.

In [ ]:
# Alignment of two different videos to each other using the template of the first video.

ref = "../data/k+_1.avi" # path to reference video file
fname = "../data/spontan_1.avi" # path to video file
framerate = 5 # frames per second of the video

##### Load both videos in CaImAn format.

In [ ]:
# Load both videos
m_orig = cma.load_video(fname=fname, fframe=framerate, play_movies=False, downsample_ratio=1, resize=False)
m_ref = cma.load_video(fname=ref, fframe=framerate, play_movies=False, downsample_ratio=1, resize=False)

##### Select the parameters for the first two motion corrections.

In [ ]:
# Create a motion correction object with parameters for analysis
max_shifts = (12, 12)   # maximum allowed rigid shift in pixels (view the movie to get a sense of motion)
strides =  (96, 96)     # create a new patch every x pixels for pw-rigid correction
overlaps = (48, 48)     # overlap between patches (size of patch strides+overlaps)
max_deviation_rigid = 6 # maximum deviation allowed for patch with respect to rigid shifts
pw_rigid = False        # flag for performing rigid or piecewise rigid motion correction
shifts_opencv = True    # flag for correcting motion using bicubic interpolation (otherwise FFT interpolation is used)
border_nan = 'copy'     # replicate values along the boundary (if True, fill in with NaN)

##### For best results the template and the input video should both be motion corrected first.

In [ ]:
# Run motion correction on both videos to correct for motion artifacts
m_ref_corr = cma.run_motioncorrect(m_ref, max_shifts=max_shifts, strides=strides, overlaps=overlaps, 
                                                  max_deviation_rigid=max_deviation_rigid, shifts_opencv=shifts_opencv, border_nan=border_nan, pw_rigid=pw_rigid)
m_orig_corr = cma.run_motioncorrect(m_orig, max_shifts=max_shifts, strides=strides, overlaps=overlaps, max_deviation_rigid=max_deviation_rigid, shifts_opencv=shifts_opencv, border_nan=border_nan, pw_rigid=pw_rigid)  

Running rigid motion correction.


100%|██████████| 1/1 [00:01<00:00,  1.85s/it]


Motion correction done. Returning the corrected movie.
Running rigid motion correction.


100%|██████████| 1/1 [00:00<00:00,  1.97it/s]


Motion correction done. Returning the corrected movie.


##### After successfull motion correction the input video can be aligned to the template video. 

In [ ]:
# Run motion correction with alignment of the two videos to each other using the template 
# of the first video by setting align to True and providing the reference video as template.
m_orig_aligned = cma.run_alignment(orig=m_orig_corr, ref=m_ref_corr, max_shifts=max_shifts, strides=strides, overlaps=overlaps, max_deviation_rigid=max_deviation_rigid, 
                               shifts_opencv=shifts_opencv, nonneg_movie=True, border_nan=border_nan, save_movie=True)

      133761 [timeseries.py:                save():300] [33902] No file saved
100%|██████████| 1/1 [00:00<00:00,  1.21it/s]


Alignment of the two videos done. Returning the aligned movie, the motion corrected reference video and the motion corrected original video.


##### Finally, all three adjusted videos can be saved as .avi files.

In [ ]:
# Export of the file as .avi / .nwb / .tif
m_ref_corr.save(file_name='./export/m_corr_k.avi', 
            q_min=0.0, 
            q_max=252.0)

m_orig_corr.save(file_name='./export/m_corr_spon.avi', 
            q_min=0.0, 
            q_max=252.0)

m_orig_aligned.save(file_name='./export/m_aligned_spon.avi', 
            q_min=0.0, 
            q_max=252.0)

'./export/m_aligned_spon.avi'